# Random Neural Network Output Distribution

This notebook demonstrates passing a fixed input through a randomly initialized neural network 1500 times. We store each output, delete the model, and then plot the distribution to see what curve we get.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Define fixed input (2 values)
fixed_input = torch.tensor([[3141., 1254.13, 425.]])
outputs = []

# Run 1500 times
for i in range(2000):
    # Initialize a normal neural network
    model = nn.Sequential(
        nn.Linear(3, 128),
        nn.ReLU(),
        nn.Linear(128, 128),
        nn.ReLU(),
        nn.Linear(128, 1)
    )
    
    # Pass fixed point through the network
    with torch.no_grad():
        out = model(fixed_input)
        
    outputs.append(out.item())
    
    # Delete the model
    del model

In [ ]:
# Plot the array of numbers to see what curve we get
plt.figure(figsize=(10, 6))
plt.hist(outputs, bins=50, density=True, alpha=0.6, color='skyblue', edgecolor='black', label='Histogram')

# Plot KDE (Kernel Density Estimate) to better visualize the continuous curve
from scipy.stats import gaussian_kde
kde = gaussian_kde(outputs)
x_range = np.linspace(min(outputs), max(outputs), 500)
plt.plot(x_range, kde(x_range), color='red', lw=2, label='Density Curve')

plt.title('Distribution of Outputs from some Randomly Initialized NNs')
plt.xlabel('Output Value')
plt.ylabel('Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import random
outputs = []
for i in range(2000):
    total = 0
    for j in range(2000):
        total += random.random()
    outputs.append(total/2000)

### Why do we see a Bell Curve?
What you are seeing is a very famous mathematical phenomenon in action: **The Central Limit Theorem (CLT)**.

#### 1. The Output Layer is a Massive "Sum"
Look at the final layer: `nn.Linear(128, 1)`. Mathematically, the single output value ($Y$) is calculated by taking the 128 activations from the previous layer ($a_1, a_2... a_{128}$), multiplying each by a randomly initialized weight ($w_1, w_2... w_{128}$), and summing them all up along with a bias term ($b$).

The formula looks like this:
$$Y = (w_1 \cdot a_1) + (w_2 \cdot a_2) + ... + (w_{128} \cdot a_{128}) + b$$

#### 2. The Central Limit Theorem in Action
The **Central Limit Theorem** is a fundamental principle in statistics. It states that if you take a large number of independent random variables and add them together, their normalized sum will always **converge to a Normal Distribution (a Bell Curve)**.

Remarkably, this happens *regardless* of the underlying distribution of the individual variables! By default, PyTorch initializes weights using a Uniform distribution. However, because your final layer is summing together **128** of these random pathways, the Central Limit Theorem takes over, and the outcome of that massive sum morphs into a perfect Gaussian bell curve.

#### 3. Mean of Zero
If you look at the center of your x-axis, the peak of the bell curve is precisely at **0**. This happens because PyTorch initializes weights with a mean of `0.0` (positive and negative weights are equally likely). Therefore, the expected value of that massive sum is also `0.0`.

#### 4. Deep Learning Theory (A Bonus Fact!)
This exact observation is the foundation of a huge sub-field in AI research. In 1996, researcher Radford Neal proved mathematically that **as the width of a neural network's hidden layers goes to infinity, a randomly initialized neural network becomes mathematically equivalent to a Gaussian Process.**

If you want to break the bell curve, try changing your layer widths from `128` to something extremely small, like `nn.Linear(3, 2) -> nn.Linear(2, 2) -> nn.Linear(2, 1)`. When the sum is no longer "large enough", the Central Limit Theorem breaks down, and the bell curve will look distorted or vanish entirely!